In [1]:
!pip install openai
# LlamaIndex (for indexing strategies module)
!pip install llama-index>=0.12.0
!pip install llama-index-embeddings-openai
!pip install llama-index-llms-openai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.


In [13]:
import json
import os
import httpx

# LlamaIndex core components
from llama_index.core import (
    VectorStoreIndex,    # Standard embedding-based index
    SummaryIndex,        # Full document storage, LLM-based relevance
    TreeIndex,           # Hierarchical summarization tree
    KeywordTableIndex,   # Inverted keyword index
    Document,            # Document wrapper with text + metadata
    Settings             # Global configuration
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

In [15]:
# Import the userdata module from google.colab
from google.colab import userdata

# Fetch the OPENAI_API_KEY from Colab Secrets
OPENAI_API_KEY_FROM_SECRETS = userdata.get('OPENAI_APIKEY')

# You can then use this variable in your configurations
print(f"Successfully fetched API Key from Colab Secrets: {OPENAI_API_KEY_FROM_SECRETS is not None}")
# Note: Avoid printing the actual key in production environments for security.


Successfully fetched API Key from Colab Secrets: True


In [16]:
OPENAI_EMBEDDING_MODEL= 'text-embedding-3-small'

# =============================================================================
# CONFIGURE LLAMAINDEX SETTINGS
# =============================================================================


Settings.embed_model = OpenAIEmbedding(
    model= OPENAI_EMBEDDING_MODEL,
    api_key=OPENAI_API_KEY_FROM_SECRETS,
    timeout=120,      # 2 min timeout for embedding calls
    max_retries=5     # Retry on failure
)

print("\n" + "="*80)
print("embed_model created")
print("="*80)

Settings.llm = OpenAI(
    model='gpt-4o-mini',
    api_key=OPENAI_API_KEY_FROM_SECRETS,
    timeout=300,      # 5 min timeout (Tree/Keyword indexes are slow!)
    max_retries=5
)

print("\n" + "="*80)
print("llm created")
print("="*80)


embed_model created

llm created


In [18]:
# ============================================================================
# LOAD DATA
# ============================================================================

print("\n" + "="*80)
print("Loading Support Tickets")
print("="*80)

with open('./sample_data/synthetic_tickets.json', 'r', encoding='utf-8') as f:
    tickets = json.load(f)



Loading Support Tickets


In [20]:
# -----------------------------------------------------------------------------
# Convert to LlamaIndex Documents
# -----------------------------------------------------------------------------
# LlamaIndex uses Document objects (similar to LangChain's Document)
# Each Document has:
#   - text: The content to index
#   - metadata: Associated data for filtering/display
# -----------------------------------------------------------------------------


documents = []
for ticket in tickets:
    # Combine all fields into content (rich context for embedding)
    # IMPORTANT: Include ticket_id in text so keyword index can find it!
    content = f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
Category: {ticket['category']}
Priority: {ticket['priority']}"""

    doc = Document(
        text=content,
        metadata={
            'ticket_id': ticket['ticket_id'],
            'category': ticket['category'],
            'priority': ticket['priority'],
            'title': ticket['title']
        }
    )
    documents.append(doc)

print(f"✓ Loaded {len(documents)} support tickets")

✓ Loaded 22 support tickets


In [23]:
# ============================================================================
# PART 1: Vector Index (Flat Index)
# ============================================================================

# HOW IT WORKS:
# ─────────────
#   Build time:
#     Document → [Chunk] → [Embed] → Store vector in index
#
#   Query time:
#     Query → [Embed] → [Find nearest vectors] → Return top-K documents

# PROS:
#   ✓ Simple and effective for most use cases
#   ✓ Fast similarity search (O(1) with approximate methods)
#   ✓ Captures semantic meaning (synonyms, paraphrases)
#
# CONS:
#   ✗ No hierarchical structure
#   ✗ May return fragmented chunks (lost context)
#   ✗ All documents treated equally (no importance weighting)

# ============================================================================
print("\n" + "="*80)
print("PART 1: Vector Index (Flat Index)")
print("="*80)

print("\nVector indexing embeds all chunks and retrieves by semantic similarity.")
print("✓ Simple and effective for most use cases")
print("✓ Fast similarity search with vector databases")
print("✗ No hierarchical structure")
print("✗ May return fragmented chunks\n")

# -----------------------------------------------------------------------------
# Build the Vector Index
# -----------------------------------------------------------------------------
# from_documents() handles:
#   1. Chunking (if needed - our docs are small so no chunking)
#   2. Embedding each chunk via Settings.embed_model
#   3. Storing vectors in memory (or a vector store if configured)
# -----------------------------------------------------------------------------

vector_index = VectorStoreIndex.from_documents(documents)
print("✓ Created vector index")


PART 1: Vector Index (Flat Index)

Vector indexing embeds all chunks and retrieves by semantic similarity.
✓ Simple and effective for most use cases
✓ Fast similarity search with vector databases
✗ No hierarchical structure
✗ May return fragmented chunks

✓ Created vector index


In [26]:
# Test query - we'll use this across all index types
query = "How do I fix authentication issues after password reset?"
print(f"\nTest Query: '{query}'")

# -----------------------------------------------------------------------------
# Create Query Engine
# -----------------------------------------------------------------------------
# as_query_engine() wraps the index for easy querying:
#   - similarity_top_k=3: Return top 3 most similar documents
#   - The engine handles: embed query → search → synthesize response
# -----------------------------------------------------------------------------
vector_query_engine = vector_index.as_query_engine(similarity_top_k=3)

print("✓ Created vector query engine")

vector_response = vector_query_engine.query(query)

print("\nVector Index Results:")
print(f"Answer: {vector_response.response}\n")
print("Source Documents:")

for i, node in enumerate(vector_response.source_nodes, 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    print(f"   Score: {node.score:.4f}")  # Similarity score (higher = more similar)
    print(f"   {node.text[:150]}...")



Test Query: 'How do I fix authentication issues after password reset?'
✓ Created vector query engine

Vector Index Results:
Answer: To fix authentication issues after a password reset, clear all active sessions and force re-authentication for users. Additionally, implement automatic session cleanup whenever a password is changed to prevent similar issues in the future.

Source Documents:

1. TICK-001
   Score: 0.6258
   Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing ...

2. TICK-011
   Score: 0.4343
   Ticket ID: TICK-011
Title: SSO authentication broken after upgrade
Description: Single Sign-On integration with corporate SAML provider failing. Users...

3. TICK-016
   Score: 0.4181
   Ticket ID: TICK-016
Title: Two-factor authentication codes not working
Description: Users unable to log in with TOTP codes from authenticator apps. Co...


In [28]:
# ============================================================================
# PART 2: Summary Index
# ============================================================================

# HOW IT WORKS:
# ─────────────
#   Build time:
#     Just store all documents as-is (no embedding!)
#
#   Query time:
#     For EACH document, ask LLM: "Is this relevant to the query?"
#     Collect relevant docs → Synthesize answer
#
# WHEN TO USE:
#   - Small collections (<50 documents)
#   - Need full document context (not fragments)
#   - High-level summarization queries
#   - When semantic similarity might miss nuanced relevance
#
# WHEN NOT TO USE:
#   - Large datasets (doesn't scale!)
#   - Real-time applications (too slow)
# ============================================================================

print("\n" + "="*80)
print("PART 2: Summary Index")
print("="*80)

print("\nSummary indexing searches through document summaries/titles.")
print("✓ Good for high-level queries")
print("✓ Returns full documents, not fragments")
print("✗ Slower for large datasets (linear scan)")
print("✗ No vector similarity search\n")

# -----------------------------------------------------------------------------
# Build the Summary Index
# -----------------------------------------------------------------------------
# NOTE: No embeddings are generated!
# Documents are stored as-is for sequential LLM evaluation
# -----------------------------------------------------------------------------

summary_index = SummaryIndex.from_documents(documents)
print("✓ Created summary index")

# -----------------------------------------------------------------------------
# Query Engine with Tree Summarize
# -----------------------------------------------------------------------------
# response_mode="tree_summarize":
#   1. Collects all relevant documents
#   2. If too many, summarizes in groups
#   3. Combines group summaries into final answer
#
# Good for comprehensive answers that synthesize multiple sources

# -----------------------------------------------------------------------------
summary_query_engine = summary_index.as_query_engine(response_mode="tree_summarize")
print("✓ Created summary query engine")

summary_response = summary_query_engine.query(query)
print("\nSummary Index Results:")
print(f"Answer: {summary_response.response}\n")
print("Source Documents:")
for i, node in enumerate(summary_response.source_nodes[:3], 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    print(f"   {node.text[:150]}...")


PART 2: Summary Index

Summary indexing searches through document summaries/titles.
✓ Good for high-level queries
✓ Returns full documents, not fragments
✗ Slower for large datasets (linear scan)
✗ No vector similarity search

✓ Created summary index
✓ Created summary query engine

Summary Index Results:
Answer: To resolve authentication issues after a password reset, clear all active sessions and force users to re-authenticate. Additionally, implement automatic session cleanup whenever a password is changed to prevent similar issues in the future.

Source Documents:

1. TICK-001
   Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing ...

2. TICK-002
   Ticket ID: TICK-002
Title: Database connection timeout in production
Description: Application experiencing intermittent 500 errors. Logs show 'connect...

3. TICK-003
   Ticket ID: TICK-003
Title: Payment processing fails for international

In [29]:
# ============================================================================
# PART 3: Tree Index (Hierarchical Retrieval)
# ============================================================================

#
# BEST FOR LARGE DOCUMENT COLLECTIONS WITH NATURAL HIERARCHY
#
# HOW THE TREE IS BUILT (Bottom-Up, EXPENSIVE):
# ──────────────────────────────────────────────
#   1. Each document/chunk becomes a LEAF node
#   2. Leaves are grouped SEQUENTIALLY (by insertion order!) into
#      groups of `num_children` — NOT by topic similarity!
#   3. LLM summarizes each group → creates parent SUMMARY nodes
#   4. Repeat on summaries until a single ROOT node remains

# TRADE-OFF:
#   Build time: SLOW (many LLM calls to create summaries)
#   Query time: FAST (logarithmic traversal)
#   num_children: Lower = deeper tree, finer summaries, more build cost
#                 Higher = shallower tree, coarser summaries, less build cost
#
# WHEN TO USE:
#   - Very large document collections (1000s of docs)
#   - Hierarchically structured content (books, manuals, wikis)
#   - When build time is acceptable (offline indexing)
#   - Multi-level queries (broad → narrow)
#
# ============================================================================

print("\n" + "="*80)
print("PART 3: Tree Index (Hierarchical Retrieval)")
print("="*80)

print("\n⚠️  NOTE: Tree Index makes many LLM calls during build.")
print("    Using first 10 documents to reduce API calls.")
print("    This may take 1-2 minutes. Please wait...\n")

print("Tree indexing builds a hierarchical structure from leaf to root.")
print("- Queries start at summary level, then drill down")
print("✓ Preserves document context and relationships")
print("✓ Efficient for large document collections")
print("✗ More complex to build and maintain\n")

# -----------------------------------------------------------------------------
# Build the Tree Index
# -----------------------------------------------------------------------------

# Default num_children=10 → groups of 10 docs per summary
# For 50 docs: ~5 summaries at level 1, then 1 root = ~6 LLM calls

tree_index = TreeIndex.from_documents(documents, num_children=10)
print("✓ Created tree index")

# -----------------------------------------------------------------------------
# Create Query Engine
# -----------------------------------------------------------------------------
# child_branch_factor controls how many branches the LLM follows at each level:
#
#   =1 (greedy):  Fast but may miss info in other branches
#                  Good for focused single-topic queries
#   =2 (balanced): Explores top 2 branches per level — catches multi-topic
#                  queries like "auth AND billing issues" (recommended)
#   =N (all):     Explores everything — maximum recall but slow,
#                  approaches Summary Index behavior
#
# At each level, the LLM scores ALL children for relevance to the query,
# then follows only the top `child_branch_factor` branches downward.
# -----------------------------------------------------------------------------

tree_query_engine = tree_index.as_query_engine(child_branch_factor=2)
print("✓ Created tree query engine")

tree_response = tree_query_engine.query(query)
print("\nTree Index Results:")

print(f"Answer: {tree_response.response}\n")
print("Source Documents:")
for i, node in enumerate(tree_response.source_nodes[:3], 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    print(f"   {node.text[:150]}...")



PART 3: Tree Index (Hierarchical Retrieval)

⚠️  NOTE: Tree Index makes many LLM calls during build.
    Using first 10 documents to reduce API calls.
    This may take 1-2 minutes. Please wait...

Tree indexing builds a hierarchical structure from leaf to root.
- Queries start at summary level, then drill down
✓ Preserves document context and relationships
✓ Efficient for large document collections
✗ More complex to build and maintain

✓ Created tree index
✓ Created tree query engine

Tree Index Results:
Answer: To fix authentication issues after a password reset, clear all active sessions and force users to re-authenticate. Additionally, implement automatic session cleanup whenever a password is changed to prevent similar issues in the future.

Source Documents:

1. TICK-001
   Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing ...

2. TICK-016
   Ticket ID: TICK-016
Title: Two-factor 

In [31]:
# ============================================================================
# PART 4: Keyword Table Index
# ============================================================================

# HOW IT WORKS:
# ─────────────
#   Build time:
#     For each document, extract keywords (via LLM or rules)
#     Build inverted index: keyword → [doc_ids]
#
#   Query time:
#     Extract keywords from query
#     Look up documents containing those keywords
#     Return matching documents

# ============================================================================
print("\n" + "="*80)
print("PART 4: Keyword Table Index")
print("="*80)

print("\n⚠️  NOTE: Keyword Index makes LLM calls to extract keywords.")
print("    Using first 10 documents to reduce API calls.")
print("    This may take 1-2 minutes. Please wait...\n")

print("Keyword indexing extracts keywords and uses exact/fuzzy matching.")
print("✓ No embeddings needed - works without vector DB")
print("✓ Good for keyword-specific queries")
print("✗ No semantic understanding")
print("✗ Misses synonyms and related concepts\n")

# -----------------------------------------------------------------------------
# Build the Keyword Table Index
# -----------------------------------------------------------------------------

keyword_index = KeywordTableIndex.from_documents(documents)
print("✓ Created summary index")

# Show the extracted keyword table (inverted index)
keyword_table = keyword_index.index_struct.table
print(f"\n✓ Extracted {len(keyword_table)} unique keywords:")
for keyword, node_ids in sorted(keyword_table.items()):
    print(f"  '{keyword}' → {len(node_ids)} document(s)")

# Create query engine
keyword_query_engine = keyword_index.as_query_engine()
print("\n✓ Created keyword query engine")

keyword_response = keyword_query_engine.query(query)

print("\nKeyword Index Results:")
print(f"Answer: {keyword_response.response}\n")
print("Source Documents:")
for i, node in enumerate(keyword_response.source_nodes[:3], 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    print(f"   {node.text[:150]}...")



PART 4: Keyword Table Index

⚠️  NOTE: Keyword Index makes LLM calls to extract keywords.
    Using first 10 documents to reduce API calls.
    This may take 1-2 minutes. Please wait...

Keyword indexing extracts keywords and uses exact/fuzzy matching.
✓ No embeddings needed - works without vector DB
✓ Good for keyword-specific queries
✗ No semantic understanding
✗ Misses synonyms and related concepts

✓ Created summary index

✓ Extracted 268 unique keywords:
  '17' → 1 document(s)
  '413' → 1 document(s)
  '413 error' → 1 document(s)
  '8' → 1 document(s)
  'acceptance' → 1 document(s)
  'acceptance window' → 1 document(s)
  'account' → 1 document(s)
  'accounts' → 1 document(s)
  'accuracy' → 1 document(s)
  'aggressive' → 1 document(s)
  'alerts' → 2 document(s)
  'algorithm' → 1 document(s)
  'anonymization' → 1 document(s)
  'api' → 2 document(s)
  'apis' → 1 document(s)
  'app' → 1 document(s)
  'apps' → 1 document(s)
  'async' → 1 document(s)
  'async/await' → 1 document(s)
  '